<a href="https://colab.research.google.com/github/KarimBekhtiGalvao/Diffusion-Based-Generation/blob/main/Diffusion_Based_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Report - Diffusion Based Text Generation
##**Subject chosen:**
Most modern text generation systems are based on autoregressive Transformer models, which generate text token by token in a left-to-right fashion. While that turned out to be highly effective, this paradigm is not the only possible approach to sequence generation. In other domains, such as image, speech, and audio generation, diffusion models have shown remarkable performance by generating data through an iterative denoising process rather than autoregressive decoding.

Diffusion models generate samples by progressively refining a noisy signal into a final clean output. Instead of predicting the next token, they learn to reverse a noise corruption process, iteratively improving the quality of the generated sample. This non-autoregressive generation process offers potential advantages, such as parallel generation.

Despite their success in other modalities, diffusion-based approaches have so far shown limited effectiveness in text generation, mainly due to the discrete nature of text and the difficulty of defining suitable diffusion processes over token sequences. Nevertheless, recent work suggests that diffusion models may be potentially viable.

##**Project Objective**
The goal of this project is to design, implement, and evaluate diffusion-based models for text generation, and to compare their behavior and performance with standard autoregressive Transformer models. The focus of the project is not on achieving state-of-the-art performance, but on understanding the strengths and limitations of diffusion models for text, analyzing their training dynamics and generation behavior, and comparing them fairly to autoregressive baselines at a small scale.

##**Proposed Project Plan**

Literature Review: Conduct a focused literature review on diffusion models for sequence and text generation. In particular, try to understand the general principles of diffusion and denoising-based generative models, have insights into existing approaches to diffusion over discrete sequences or text representations, and figure out the challenges and limitations of diffusion-based text generation.

Data and Representation: Select a suitable text dataset for small-scale experimentation. Possible choices include a subset of Wikipedia or book corpora. You may operate at the character, byte, or token level. Dataset size should be chosen to keep training computationally manageable.

Baseline Model (Autoregressive): Implement a standard autoregressive text generation baseline. Train and evaluate the model using standard text generation metrics. Tune hyperparameters using validation data.

Diffusion-Based Model: Implement one or more diffusion-based models for text generation. Define an appropriate forward noise process and a reverse denoising model. Clearly explain architectural and modeling choices. Ensure that training and inference are computationally feasible.

Evaluation and Analysis: Compare diffusion-based models and autoregressive models using quantitative metrics (e.g., perplexity, negative log-likelihood, or other suitable measures). Do a qualitative analysis of the generated text, training stability, generation speed, and other relevant aspects you might find relevant. Optionally, you may explore scaling behavior, such as how performance changes with model size or dataset size for diffusion-based versus autoregressive models.



# Litterature Review

## Introduction

As seen during the COMP6861 Concordia course, generating text using Machine Learning is a complex problem, with the modeling of high-dimensional probability distributions over sequences of tokens. Approaches seen in class rely on autoregressive language models that generate text by predicting the next token using previously generated tokens. While these models have achieved strong performances such as that of Claude or ChatGPT 5, they suffer from problems such as **exposure bias**, **limited controllability**, and **difficulties modeling long-range dependencies**.

More recently, diffusion-based generative models have emerged as a promising alternative paradigm for generative modeling [Deep Unsupervised Learning using Nonequilibrium Thermodynamics, Sohl-Dickstein et al]. Used initially to generate images, diffusion models learn to make sentences by gradually transforming noise into structured data through a sequence of denoising steps. The process typically works in two stages: a forward process, which uses training data and noises it, and a denoising process, where a neural network learns to remove noise and reconstruct the initial data distribution [Diffusion models in text generation: a survey].

Diffusion models have been successful in image generation so recent research has begun adapting them to discrete data, including natural language. Applying diffusion models to text introduces issues because of the discrete nature of tokens/text and the difficulty of defining continuous noise processes over discrete representations. To solve these problems, several approaches have been proposed, including continuous embeddings [A CHEAPER AND BETTER DIFFUSION LANGUAGE MODEL WITH SOFT-MASKED NOISE], discrete diffusion processes, and hybrid autoregressive–diffusion architectures [AR-DIFFUSION: Auto-Regressive Diffusion Model for Text Generation].

This line of research shows that diffusion-based generation may offer advantages such as diversity, better coherence, and more flexible generation compared to traditional models. Let's dive by using the survey [Diffusion models in text generation: a survey] which provides a timelime of recent influential models : [Diffusion-NAT], [AR-DIFFUSION], [DiffuSum], [Masked-Diffuse LM], [DiNoiSer] and [RDMs].

# Why diffusion ?

Models studied in class are Auto-regressive, meaning they take a token as an input and generates a token as a result, with the idea that a well trained model should output a token making sense after that. It stems from the idea that the next word coming in a sentence depends on the words before.

However, there might be languages where sentences are not build left to right and where the last word of a sentence modulates the beginning of it. In such cases, it could be interesting to use models that are not auto-regressive.

Diffusion are such models, and take a fully noised sequence (think: noisy image) and denoise it iteratively to come to an ensemble making sense.

Indeed, diffusion for Machine Learning models is a domain that really took off for images, but gets less scrutiny for LLMs than autoregressive models.

This use of diffusion in ML has been first used in [Denoising Diffusion Probabilistic Models]. On images, it uses pixels value (which are easier since continuous). We will present later how to adapt that a non-continuous space like Natural Language Processing.


# Domain Inception

In [Deep Unsupervised Learning using Nonequilibrium Thermodynamics], Sohl-Dickstein et al launched the domain of using diffusion models for text generation.

They propose a model that allows: "extreme flexibility", "exact sampling", and the possibility to cheaply evaluate the model using log-likelihood, using a Markov chain.

As stated before, they define a forward diffusion process, and then learn how to reverse this process (We called it backward in the Introcution, as used in [RDMS]).

$$q(x^{(0,...,T)})=q(x^{0})\Pi_{t=1}^{T}q(x^{(t)}|x^{(t-1)})$$

After having applied this noise on the data, we train a backward process that should  take $q(x^{(0,...,T)})$ as input data and retrieve $q(x^{0})$ through T steps too:

$$p(x^{(0,...,T)})=p(x^{T})\Pi_{t=1}^{T}p(x^{(t-1)}|x^{(t)})$$

We notice that this time we start at step T and go back through the steps ($x^{(t-1)}|x^{(t)}$) to retrieve original distribution.

In [Deep Unsupervised Learning using Nonequilibrium Thermodynamics], the training is done by maximizing the log-likelihood of the model.
Amongst the serveral datasets used for evaluation, MNIST has been used. The log likelihood of the diffusion model reached 317 bits compared to 349 for the perfect model.

Following these promising results, the method has been readapted to several models. However, this model is still used on images rather than text.

In [Denoising Diffusion Probabilistic Models], diffusion is used via markov chains. It is trained using images that are noised. It behaves much like particles in a N dimensional space (with N the number of pixels) diffusing via T steps in this space. We move the value of some pixels in the space, iteratively. This is the forward process (Like particles, each pixel value diffuse in the space). The model is trained from the noised/diffused version of the image and has to bring it back to its denoised version through these T steps. This process is the backwards process.

To model the forward process, the authors of [Denoising Diffusion Probabilistic Models] add a gausian noise to every pixels. For the backward process, they make their model predict the noise that was added at this step, thus learning a local denoising function for the current step.

The question that can arise is: how do you adapt a continuous process as the one presented before to a discrete space like Natural Language Processing ?

That's what [D3PM] sets to do. They reuse the same exemple as [Deep Unsupervised Learning using Nonequilibrium Thermodynamics] (Swiss rolls), but this time dicretizing the processes.

For K categories, K and integer, they define a transition matrix Q of $ K\times K$ dimensions. This matrix reprensents the process of going from a state i to a state j. To take an exemple, if we consider the set of tokens being the Latin Alphabet, there are 26 letters so $K = 26$. The entrey $Q_{i=1,j=1}$ would reprensent the probability of going from a token $a$ to the token $a$ during the noising process.

However, it leads to the problem of size: for a Vocabulary of 1,000 tokens leads to matrices of size $1,000 \times 1,000$. They have for that defined rules of transition, revolving around two probabilities: we have a probability $1-\beta$ of not changing token and a probability $\beta$ of changing token.

This noising happens by choosing other tokens, modelizing the discretness of the space: we don't move through the embeddings but directly from tokens to tokens.

Due to the success of BERT which uses masks, another approach was formulated where when a token is corrupted, it is turned to $[MASK]$ instead.

The setup they use is that of a CNN (UNet) with Transformer sinusoidal embeddings on two tasks : character level generation with the dataset text8 with 27 tokens and LM1B, a dataset with a billion words in training data.

With this setup they achieved strong results in text generation and image generation, "comparable to autoregressive model of the same size".

# RDMs

RDM, standing for Reparameterized Discrete diffusion Models, presented in the paper [A Reparameterized Discrete Diffusion Model for Text Generation] by Zheng et al, argues that it exists and equivalent reparametrization of the sampling used previously in [Deep Unsupervised Learning using Nonequilibrium Thermodynamics] by Sohl-Dickstein et al.
This reparameterized sequence follows rules at each step (route-and-denoise process):
- has a probability of being denoised
- has a probability of being reseted to a noisy state following the routing mechanism

This process is used to noise directly the token embeddings of words, instead of adding it. It enables to stay in the token space to train the model. The route-denoise process allows to make the training process stable.

The forward process replaces tokens with other tokens from a noise distribution. A transition matrix stores the probability of each token to be corrupted into another token, and the denoising process rebuilds the orignal sentence.

The architecture used (cf annex F) predicts a target length. The input of the model is the timestep embedding (the sequence's current t variable, cf the froward/backward processes), the t is projected using sinusoidal embedinggs then passes through a two-layer MLP. We concatenate to this embedding the token embedding.

This input goes through a Transformer and then is projected to the vocabulary space to determine which word is predicted.

This work is important because it allows to use a continuous model on a discrete problem such as natural language processing. It allows to generate text too using a first sentence to which is appended noise.


# Data and representation

Accross the litterature, we were able to find several datasets. We choose as a Dataset books from the British writer Jane Austen, which allow consistentcy in the text, style, and vocabulary used.
The books are extracted from the public library of Project Gutenberg, and preprocessed to make them usable.

First we process the tecxs by getting read of the copyright and table of content notes.

# Baseline model

# Generation Model

The first model we will try to replicate will be the D3PM model for text generation [D3PM] which is the continuation of [DDPM] in a discrete manner. [DDPM] follows teh backbone of PixelCNN++ [PixelCNN++]. The authors of [D3PM] provide a github for their implementation: https://github.com/
hojonathanho/diffusion.
We use it and work on it





In [ ]:
import functools

import fire
import numpy as np
#import tensorflow.compat.v1 as tf

#from diffusion_tf import utils
from diffusion_tf.diffusion_utils import get_beta_schedule, GaussianDiffusion
#from diffusion_tf.models import unet
#from diffusion_tf.tpu_utils import tpu_utils, datasets


class Model():
  def __init__(self,
               *,
               model_name,
               betas: np.ndarray,
               loss_type: str,
               num_classes: int,
               dropout: float,
               randflip,
               block_size: int):
    self.model_name = model_name
    self.diffusion = GaussianDiffusion(betas=betas, loss_type=loss_type)
    self.num_classes = num_classes
    self.dropout = dropout
    self.randflip = randflip
    self.block_size = block_size

  def _denoise(self, x, t, y, dropout):
    B, H, W, C = x.shape.as_list()
    assert x.dtype == tf.float32
    assert t.shape == [B] and t.dtype in [tf.int32, tf.int64]
    assert y.shape == [B] and y.dtype in [tf.int32, tf.int64]
    orig_out_ch = out_ch = C

    if self.block_size != 1:  # this can be used to reduce memory consumption
      x = tf.nn.space_to_depth(x, self.block_size)
      out_ch *= self.block_size ** 2

    y = None
    if self.model_name == 'unet2d16b2c112244':  # 114M for block_size=1
      out = unet.model(
        x, t=t, y=y, name='model', ch=128, ch_mult=(1, 1, 2, 2, 4, 4), num_res_blocks=2, attn_resolutions=(16,),
        out_ch=out_ch, num_classes=self.num_classes, dropout=dropout
      )
    else:
      raise NotImplementedError(self.model_name)

    if self.block_size != 1:
      out = tf.nn.depth_to_space(out, self.block_size)
    assert out.shape == [B, H, W, orig_out_ch]
    return out

  def train_fn(self, x, y):
    B, H, W, C = x.shape
    if self.randflip:
      x = tf.image.random_flip_left_right(x)
      assert x.shape == [B, H, W, C]
    t = tf.random_uniform([B], 0, self.diffusion.num_timesteps, dtype=tf.int32)
    losses = self.diffusion.p_losses(
      denoise_fn=functools.partial(self._denoise, y=y, dropout=self.dropout), x_start=x, t=t)
    assert losses.shape == t.shape == [B]
    return {'loss': tf.reduce_mean(losses)}

  def samples_fn(self, dummy_noise, y):
    return {
      'samples': self.diffusion.p_sample_loop(
        denoise_fn=functools.partial(self._denoise, y=y, dropout=0),
        shape=dummy_noise.shape.as_list(),
        noise_fn=tf.random_normal
      )
    }

  def samples_fn_denoising_trajectory(self, dummy_noise, y, repeat_noise_steps=0):
    times, imgs = self.diffusion.p_sample_loop_trajectory(
      denoise_fn=functools.partial(self._denoise, y=y, dropout=0),
      shape=dummy_noise.shape.as_list(),
      noise_fn=tf.random_normal,
      repeat_noise_steps=repeat_noise_steps
    )
    return {
      'samples': imgs[-1],
      'denoising_trajectory_times': times,
      'denoising_trajectory_images': imgs
    }

  def interpolate_fn(self, dummy_noise, y):
    x1, x2, lam, x_interp, t = self.diffusion.interpolate(
      denoise_fn=functools.partial(self._denoise, y=y, dropout=0),
      shape=dummy_noise.shape.as_list(),
      noise_fn=tf.random_normal,
    )
    return {
      'x1': x1,    # placeholder
      'x2': x2,    # placeholder
      'lam': lam,  # placeholder
      't': t,      # placeholder
      'x_interp': x_interp
    }
